# ClinGuard-DiLoCo: Kaggle session 7 (gate G3: the diagnosis prompt)

Session 6 showed that a report writer trained on reports ignores what the taught vision side sees. This session
splits the job (decision I32, the diagnosis prompt of FedMRG / PromptMRG):

1. **Finding classifier** (`scripts/classify.py`, about 15 min): every study's X-ray goes once through the taught
   vision side; one logistic regression per finding (8 findings) learns from the training studies, and a cut-off
   per finding is set on the calibration studies (a finding is named only where that is more often right than wrong).
2. **Writer**: as in session 6, plus one sentence in the prompt, e.g. *"Findings suggested by an image classifier:
   cardiomegaly."* In training the sentence lists the report's own findings; the development reports are written
   with the classifier's predictions, and once more with the true findings (a diagnostic: does the writer follow it?).

| GPU | Jobs, in order |
|---|---|
| 0 | classifier `eyes_v1` -> `g3_A_prompted`: train on site A, write development reports (predicted, then true findings) |
| 1 | `g3_B_prompted`: train on site B, write development reports (waits for the classifier if needed) |

Estimate: about 2.5 hours. Uploads to the private HF repo with a lease; after a crash, run the notebook again and
each job continues where it stopped (finished jobs are skipped).

**Settings** (right-hand panel): GPU **T4 x2**, Internet **on**, input dataset **raddar/chest-xrays-indiana-university**,
secret **HF_TOKEN** attached. Keep the notebook **private**.

**How to run (important):** click **Save Version** (top right) -> choose **Save & Run All (Commit)** -> **Save**.
Then you can close the tab: a saved version keeps running for up to 12 h. Do **not** run the cells by hand in
the editor; that "Draft Session" is switched off when the tab is idle. When the version has finished, open it
from the notebook's version history and send the output of the last cell.

In [ ]:
HF_REPO = "tasbid66/clinguard-runs"
GIT_REF = "main"     # a resumed run refuses a different commit; to resume after main moved, set the commit printed in its log
import glob, os, subprocess, sys, time, json
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf /kaggle/working/ClinGuard && git clone -q https://github.com/tasbidrahman10/ClinGuard.git /kaggle/working/ClinGuard
%cd /kaggle/working/ClinGuard
!git checkout -q {GIT_REF} && git log --oneline -1
!pip install -q "transformers==5.23.0" "peft==0.22.0" "accelerate==1.16.0" bitsandbytes pyyaml
!pip install -q --no-deps -e .
src = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/iu_xray"):
    os.symlink(src, "data/raw/iu_xray")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

Download the 3B model once, so the two GPUs do not race for it.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/ClinGuard/src")
from huggingface_hub import snapshot_download
from clinguard.model import load
snapshot_download(load.MODEL_ID, revision=load.MODEL_REVISION)
print("model cached")

## Launch: each GPU runs its jobs one after another

In [ ]:
import os, subprocess
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
os.makedirs("/kaggle/working/logs", exist_ok=True)
CFG = "configs/experiment"
RUN = f"python scripts/run_job.py --hf-repo {HF_REPO} --work-dir /kaggle/working/runs --config {CFG}"
CLASSIFY = (f"python scripts/classify.py --hf-repo {HF_REPO} --config {CFG}/g3_A_prompted.yaml --name eyes_v1"
            " --work-dir /kaggle/working/runs")
queues = {0: [("classifier eyes_v1", CLASSIFY),
              ("train g3_A_prompted", f"{RUN}/g3_A_prompted.yaml")],
          1: [("train g3_B_prompted", f"{RUN}/g3_B_prompted.yaml")]}
procs = {}
for gpu, jobs in queues.items():
    log = open(f"/kaggle/working/logs/gpu{gpu}.log", "a")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    # '&&': a job runs only if the one before it finished
    procs[gpu] = subprocess.Popen(" && ".join(cmd for _, cmd in jobs), shell=True, stdout=log,
                                  stderr=subprocess.STDOUT, env=env)
    print("GPU", gpu, "started:", " -> ".join(label for label, _ in jobs))

## Wait, printing progress every 5 minutes

In [ ]:
import time
def tail(path, n=3):
    lines = [l for l in open(path, errors="replace").read().splitlines()
             if l.strip() and "Warning" not in l and "warn" not in l]
    return lines[-n:]
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {int(time.time() - start) // 60} min")
    for gpu in procs:
        state = "running" if procs[gpu].poll() is None else f"exit {procs[gpu].returncode}"
        print(f"GPU {gpu} ({state}):", *tail(f"/kaggle/working/logs/gpu{gpu}.log"), sep="\n   ")
print({gpu: p.returncode for gpu, p in procs.items()})

## Summary (self-contained: after a restart, run cell 2 and then this cell)

In [ ]:
import json, os, sys
from huggingface_hub import hf_hub_download
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")
import run_job as rj
import classify
from huggingface_hub import HfApi
def fetch(path):
    try:
        return hf_hub_download(HF_REPO, path, force_download=True)
    except Exception as e:
        print("missing:", path, type(e).__name__); return None
files = set(HfApi().list_repo_files(HF_REPO))
for n in ("g3_A_prompted", "g3_B_prompted"):
    r = rj.run_id_for(rj.load_config(f"configs/experiment/{n}.yaml"))
    lease = fetch(f"runs/{r}/LEASE.json")
    print(f"{n:14s} {r}  status: {json.load(open(lease))['status'] if lease else '-'}")
    rounds = fetch(f"runs/{r}/rounds.jsonl")
    if rounds:
        rows = [json.loads(l) for l in open(rounds)]
        site = next(iter(rows[0]["train_loss"]))
        print("   train loss by round:", " ".join(f"{x['train_loss'][site]:.3f}" for x in rows),
              f"| {rows[-1]['cost'][site]['train_sec'] / 100:.2f} s/step")
    for folder in ("generations", "generations_oracle"):
        have = sorted(f.rsplit("/", 1)[-1] for f in files if f.startswith(f"runs/{r}/{folder}/"))
        print(f"   {folder}: {', '.join(have) or 'none yet'}")
report = fetch("classifier/eyes_v1/report.json")
if report:
    classify.print_report(json.load(open(report)))

**Send Tasbid / Claude:** the output of the last cell. The development reports stay in the HF repo; they are
scored locally in the evaluation environment (with the spotter's findings, and with the true findings).